# API — Akuisisi Data Entitas Budaya & Museum Bali via REST API Wikidata
**Nama:** Yuliana Farida | **NIM:** 2501010372 | **Email:** faridayuli0903@gmail.com

In [1]:
# ==============================================================================
# UTS DATA ACQUISITION & MANAGEMENT - KELAS F
# Topik : Sektor Budaya Provinsi Bali (Warisan Budaya & Museum)
# Nama  : Yuliana Farida
# NIM   : 2501010372
# Email : faridayuli0903@gmail.com
# Prodi : Informatika - INSTIKI
# ==============================================================================
import os, re, io, sys, json, time, shutil, hashlib, datetime as dt
from pathlib import Path
import requests
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from bs4 import BeautifulSoup
from urllib.robotparser import RobotFileParser

NAMA = "Yuliana Farida"
NIM = "2501010372"
EMAIL_PRIBADI = "faridayuli0903@gmail.com"
UA = f"INSTIKI-UTS-DAM-BUDAYA/1.0 ({EMAIL_PRIBADI}; {NAMA}; NIM {NIM})"
TIMEOUT = 25
JEDA = 2.0

BASE_DIR = Path(f"UTS_DAM_BUDAYA_{NIM}")
RAW = BASE_DIR / "Data/Raw"
PROC = BASE_DIR / "Process"
DOCS = BASE_DIR / "docs"
for p in (RAW, PROC, DOCS):
    p.mkdir(parents=True, exist_ok=True)

PROV_FILE = DOCS / "provenance.csv"

def hitung_md5(filepath):
    h = hashlib.md5()
    with open(filepath, "rb") as f:
        for chunk in iter(lambda: f.read(8192), b""):
            h.update(chunk)
    return h.hexdigest()

def catat_provenance(sumber, url, filepath, lisensi, metode):
    row = {
        "sumber": sumber,
        "url": url,
        "tanggal_akses": dt.datetime.now().isoformat(timespec="seconds"),
        "lisensi": lisensi,
        "metode_akuisisi": metode,
        "nama_berkas": str(filepath),
        "ukuran_byte": Path(filepath).stat().st_size,
        "md5_hash": hitung_md5(filepath)
    }
    df_baru = pd.DataFrame([row])
    if PROV_FILE.exists():
        df_lama = pd.read_csv(PROV_FILE)
        df_baru = pd.concat([df_lama[df_lama["sumber"] != sumber], df_baru], ignore_index=True)
    df_baru.to_csv(PROV_FILE, index=False)

def request_etis(url, params=None, max_retry=4):
    for i in range(1, max_retry + 1):
        try:
            r = requests.get(url, params=params, headers={"User-Agent": UA}, timeout=TIMEOUT)
            if r.status_code == 200:
                time.sleep(JEDA)
                return r
            if r.status_code in (429, 500, 502, 503, 504):
                time.sleep(2 ** i)
                continue
            r.raise_for_status()
        except (requests.Timeout, requests.ConnectionError):
            time.sleep(2 ** i)
    raise RuntimeError(f"Gagal mengambil {url}")

print("Konfigurasi Topik Budaya selesai. User-Agent:", UA)

In [1]:
# ==============================================================================
# SOAL 3: AKUISISI DATA VIA REST API WIKIDATA (100% SEKTOR BUDAYA BALI)
# ============================
==================================================
WILAYAH_BALI = [
    "Kabupaten Badung", "Kabupaten Bangli", "Kabupaten Buleleng",
    "Kabupaten Gianyar", "Kabupaten Jembrana", "Kabupaten Karangasem",
    "Kabupaten Klungkung", "Kabupaten Tabanan", "Kota Denpasar"
]

WIKIDATA_API = "https://www.wikidata.org/w/api.php"
hasil_api_budaya = []

for wil in WILAYAH_BALI:
    nama_bersih = wil.replace("Kabupaten ", "").replace("Kota ", "").strip()
    params = {
        "action": "query",
        "list": "search",
        "srsearch": f"museum OR pura OR budaya {nama_bersih} Bali",
        "srlimit": 50,
        "format": "json"
    }
    resp = request_etis(WIKIDATA_API, params=params)
    data_json = resp.json()

    raw_json_path = RAW / f"wikidata_budaya_{nama_bersih.lower()}.json"
    raw_json_path.write_text(json.dumps(data_json, ensure_ascii=False, indent=2), encoding="utf-8")

    total_entitas = data_json.get("query", {}).get("searchinfo", {}).get("totalhits", 0)
    hasil_api_budaya.append({
        "wilayah": wil,
        "kunci_wilayah": nama_bersih.lower(),
        "jumlah_entitas_wikidata": total_entitas
    })

df_api_budaya = pd.DataFrame(hasil_api_budaya)
path_api_proc = PROC / "api_wikidata_budaya_bali.csv"
df_api_budaya.to_csv(path_api_proc, index=False)

catat_provenance(
    "API_Wikidata_Budaya_Bali",
    WIKIDATA_API,
    path_api_proc,
    "CC0 1.0 Public Domain (Wikidata Action API)",
    "REST API Query"
)

print(f"Total wilayah hasil API Wikidata Budaya: {len(df_api_budaya)} baris")
display(df_api_budaya)